# Criação de Comunicação com LLM
Roteiro para criação de comunicação com LLM utilizando API do Gemini.

- [Criação via API Studio](https://aistudio.google.com/apikey)

**Obs:** Este roteiro está em Python, mas todos os links possuem versões em JavaScript!

#### Também Recomendamos o OpenRouter
- [OpenRouter: Interface geral para modelos](https://openrouter.ai/)

---
## Passo 0 — Configure seu acesso (faça agora)

**1.** Crie sua chave em https://aistudio.google.com/apikey — é gratuito e não pede cartão.

**2.** No Colab, clique na chave 🔑 na barra lateral esquerda → **Adicionar novo secret**
   - Nome: `GOOGLE_API_KEY`
   - Valor: sua chave
   - Ative o botão **Acesso ao notebook**

**3.** Rode as duas células abaixo. A segunda diz se deu certo.

> Se for rodar local, guarde suas chaves em um `.env` e coloque o `.env` no `.gitignore`. Nunca commite uma chave.

In [2]:
%pip install -q google-genai

### Setup e autodiagnóstico

In [5]:
# Setup e autodiagnóstico — rode e leia a saída

import os, time

MODEL = "gemini-3.1-flash-lite"

def carregar_chave():
    try:
        from google.colab import userdata
        return userdata.get("GOOGLE_API_KEY")
    except Exception:
        return os.environ.get("GOOGLE_API_KEY")

GOOGLE_API_KEY = carregar_chave()

def diagnosticar(msg):
    if "503" in msg or "UNAVAILABLE" in msg:
        return ("503 = o modelo está sobrecarregado no Google. Não é você.\n"
                "       Espere 1-2 min, ou troque MODEL para 'gemini-3.1-flash-lite'.")
    if "429" in msg or "RESOURCE_EXHAUSTED" in msg:
        return "429 = cota estourada. Espere ~30s e tente de novo."
    if "403" in msg or "PERMISSION" in msg:
        return "403 = chave sem permissão para este modelo."
    if "404" in msg or "NOT_FOUND" in msg:
        return "404 = modelo indisponível para esta conta. Tente 'gemini-3.1-flash-lite'."
    if "400" in msg or "API_KEY_INVALID" in msg:
        return "400 = chave inválida. Gere outra no AI Studio."
    return "Erro não catalogado — leia o detalhe abaixo."

if not GOOGLE_API_KEY:
    print("[ERRO] Chave não encontrada.")
    print("       Colab: ícone de chave na lateral -> secret GOOGLE_API_KEY -> ligue 'Acesso ao notebook'")
    print("       Local: export GOOGLE_API_KEY=sua-chave")
else:
    from google import genai
    client = genai.Client(api_key=GOOGLE_API_KEY)
    erro = None
    for tentativa in range(3):          # 503 e 429 são transitórios: tenta de novo
        try:
            r = client.models.generate_content(model=MODEL, contents="responda apenas: ok")
            print("[OK] Tudo certo. Modelo:", MODEL)
            print("     Resposta do teste:", r.text.strip())
            erro = None
            break
        except Exception as e:
            erro = str(e)
            if not any(x in erro for x in ("503", "UNAVAILABLE", "429", "RESOURCE_EXHAUSTED")):
                break
            if tentativa < 2:
                print("   instabilidade no servidor, tentando de novo...")
                time.sleep(2 * (tentativa + 1))
    if erro:
        print("[ERRO] A chamada falhou.")
        print("      ", diagnosticar(erro))
        print("       Detalhe:", erro[:300])

[OK] Tudo certo. Modelo: gemini-3.1-flash-lite
     Resposta do teste: ok


### Erros que você pode encontrar (e o que fazer)

| Erro | Significa | O que fazer |
|---|---|---|
| **400** `API_KEY_INVALID` | A chave está errada ou veio com espaço | Cole corretamente ou gere outra no AI Studio |
| **403** `PERMISSION_DENIED` | A chave não tem acesso a esse modelo | Confira o `MODEL` |
| **404** `NOT_FOUND` | O modelo não existe para a sua conta | Troque para um [modelo válido](https://ai.google.dev/gemini-api/docs/models?hl=pt-br#all-gemini-3-models) |
| **429** `RESOURCE_EXHAUSTED` | Rate limit: chamadas demais em pouco tempo | Espere ~30s |
| **503** `UNAVAILABLE` | O modelo está sobrecarregado **no Google**, comum em horários de pico | Espere 1–2 min, ou troque o `MODEL` |

Os limites do free tier mudam com o tempo e são por modelo. Os seus, ao vivo: https://aistudio.google.com/rate-limit

In [ ]:
# Opcional: só use se estiver tomando 429 ou 503 com frequência.
# Tenta de novo com espera crescente: 2s, 4s, 8s...

import time

TRANSITORIOS = ("429", "RESOURCE_EXHAUSTED", "503", "UNAVAILABLE")

def gerar_com_retry(client, model, contents, config=None, tentativas=4):
    for i in range(tentativas):
        try:
            return client.models.generate_content(model=model, contents=contents, config=config)
        except Exception as e:
            if not any(x in str(e) for x in TRANSITORIOS) or i == tentativas - 1:
                raise
            espera = 2 ** (i + 1)
            print("erro transitório. Tentando de novo em", espera, "s...")
            time.sleep(espera)

# uso: resp = gerar_com_retry(client, MODEL, "sua pergunta aqui")

Existem muitas formas de utilizar a API do Google e este arquivo aborda apenas a utilização padrão da api do gemini.
- Por exemplo, o langchain tem um wrapper sobre a API do Gemini ([Langchain ChatGoogleGenerativeAI](https://python.langchain.com/docs/integrations/chat/google_generative_ai/))
- Através do OpenRouter: ([OpenRouter: Gemini 3.1 Flash Lite](https://openrouter.ai/google/gemini-3.1-flash-lite?view=api))

## Primeira chamada
Localmente, instale via `pip install google-genai`



In [ ]:
from google import genai

client = genai.Client(api_key=GOOGLE_API_KEY)

response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents="Defina o Sport Recife em uma frase",
)

print(response.text)

O Sport Club do Recife é uma instituição centenária de paixão arrebatadora, conhecida por sua torcida fervorosa, pelo orgulho de ser o "Leão da Ilha" e por ser um gigante do futebol nordestino com uma história marcada por conquistas inesquecíveis.


> Estamos usando `generate_content`, que a Google hoje chama de *Generate Content API*
> e classifica como legada. Ela segue **totalmente suportada**, sem data de desligamento. Para projetos novos,
> a Google recomenda a *Interactions API*, que guarda o histórico no servidor.
>
> Usamos esta aqui de propósito: ela expõe o que acontece em cada chamada, que é justamente o assunto da aula.
> Os conceitos (roles, contexto, tokens, tools, saída estruturada) são os mesmos nas duas.

#### Detalhamento do Response

[Referência da API](https://ai.google.dev/api/generate-content#v1beta.GenerateContentResponse)

```
candidates=
[
  Candidate(content=Content(
  parts=[Part(video_metadata=None,
      thought=None, inline_data=None, code_execution_result=None,
      executable_code=None, file_data=None, function_call=None,
      function_response=None,
      text='O Sport Recife é um clube de futebol
      tradicional de Pernambuco, conhecido por sua camisa rubro-negra e uma
      história rica em conquistas regionais e nacionais.\n'
      )], role='model'),
  citation_metadata=None,
  finish_message=None,
  token_count=None,
  finish_reason=<FinishReason.STOP: 'STOP'>,
  url_context_metadata=None,
  avg_logprobs=-0.21691273197983252,
  grounding_metadata=None, index=None, logprobs_result=None,
  safety_ratings=None
)]
create_time=None
response_id=None
model_version='gemini-3.5-flash-lite'
prompt_feedback=None
usage_metadata=GenerateContentResponseUsageMetadata
    (
      cache_tokens_details=None,
      cached_content_token_count=None,
      candidates_token_count=33,
      candidates_tokens_details=
        [
          ModalityTokenCount(modality=<MediaModality.TEXT:
          'TEXT'>, token_count=33)
        ],
      prompt_token_count=8,
      prompt_tokens_details=
        [
          ModalityTokenCount(modality=<MediaModality.TEXT: 'TEXT'>, token_count=8)
        ],
      thoughts_token_count=None,
      tool_use_prompt_token_count=None,
      tool_use_prompt_tokens_details=None, total_token_count=41, traffic_type=None
    )
automatic_function_calling_history=[]
parsed=None
```

### Quanto custou essa chamada?

Todas as respostas trazem `usage_metadata`. **Você paga por tokens de entrada + tokens de saída**. A entrada
inclui o system prompt, os documentos anexados e **todo o histórico da conversa** que você reenviar.

É por isso que uma longa conversa encarece, já que cada mensagem nova reenvia tudo que veio antes.

In [ ]:
# resp = client.models.generate_content(model=MODEL, contents="Defina o Sport Recife em uma frase")
resp = response

u = resp.usage_metadata
print("tokens de entrada :", u.prompt_token_count)
print("tokens de saída   :", u.candidates_token_count)
print("total             :", u.total_token_count)

tokens de entrada : 9
tokens de saída   : 62
total             : 71


## Roles e Formatos de Mensagens

### Formato padrão do Gemini

No Gemini, por padrão, as mensagens se organizam com o seguinte formato:


```json
{
  "role": string
  "parts": [
    "text": string,
    "inline_data": (imagem, arquivos...)
  ]
}
```

Temos essencialmente duas roles (papéis) na troca de mensagens:
1. `user`
2. `model`

Esse é o formato considerado neste roteiro.

Para entender como estruturar melhor o Parts, visite [esta referência](https://cloud.google.com/vertex-ai/generative-ai/docs/reference/rest/v1beta1/Content).

### Formato compatível com a OpenAI

Também é disponibilizado um formato compatível com o padrão da OpenAI.

Temos essencialmente três roles (papéis) na troca de mensagens:

1. `user`
2. `assistant`
3. `system`

Abaixo um exemplo de estrutura de mensagem

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key=GOOGLE_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

response = client.chat.completions.create(
    model="gemini-3.1-flash-lite",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": "Explain to me how AI works"
        }
    ]
)

print(response.choices[0].message.content)

> **Por que isso importa na prática:** esse formato compatível com a OpenAI é o mesmo que o
> **OpenRouter** e a maioria dos provedores falam. Se a cota do Google travar no meio do seu projeto,
> você troca a `base_url` e a `api_key`. O resto do código continua igual.
>
> O conceito não muda de provedor para provedor. Só o endereço muda.

## A API não lembra de nada

Esse é o ponto que mais derruba quem está começando: **cada chamada é independente.**
O modelo não guarda a conversa anterior. Você precisa reenviar o histórico a cada chamada.

O "chat" que você conhece é uma ilusão construída por cima disso: a cada mensagem nova,
o aplicativo reenvia a conversa inteira.

> Modelos de API mais recentes, como a Responses API (OpenAI) e a Interactions API (Gemini) resolvem isso através de mecanismos de cache e lidando com o hsitórico para você, cobrando menos por conteúdo guardado em memória.

In [ ]:
# 1) Duas chamadas separadas

client.models.generate_content(model=MODEL, contents="Meu nome é Artur.")
r = client.models.generate_content(model=MODEL, contents="Qual é o meu nome?")
print("SEM histórico ->", r.text.strip())

SEM histórico -> Como sou uma inteligência artificial, não tenho acesso aos seus dados pessoais, documentos ou à sua identidade real, a menos que você tenha me dito o seu nome anteriormente nesta conversa.

Se você já me disse, eu esqueci devido às limitações da minha memória de curto prazo (cada nova conversa começa do zero). 

**Como você se chama?** Se me disser agora, poderei usar o seu nome durante o restante desta sessão!


In [ ]:
from google.genai import types

# 2) Agora reenviando o histórico

historico = [
    types.Content(role="user",  parts=[types.Part(text="Meu nome é Artur.")]),
    types.Content(role="model", parts=[types.Part(text="Prazer, Artur!")]),
    types.Content(role="user",  parts=[types.Part(text="Qual é o meu nome?")]),
]

r = client.models.generate_content(model=MODEL, contents=historico)
print("COM histórico ->", r.text.strip())

COM histórico -> Seu nome é Artur. Como posso te ajudar hoje?


In [ ]:
# 3) Na prática: um chat de verdade é só um loop que acumula o histórico

historico = []

def conversar(mensagem):
    historico.append(types.Content(role="user", parts=[types.Part(text=mensagem)]))
    r = client.models.generate_content(model=MODEL, contents=historico)
    historico.append(types.Content(role="model", parts=[types.Part(text=r.text)]))
    return "\n-------------\n" + r.text.strip() + "\n-------------\n"

print(conversar("Meu time é o Sport. Guarde isso."))
print(conversar("Qual é o meu time?"))
print("\nMensagens no histórico:", len(historico))


-------------
Combinado! Registrado: seu time é o **Sport Club do Recife**. 🦁

Estarei ciente disso para nossas futuras conversas. Precisando de qualquer informação sobre o Leão ou sobre futebol, é só chamar!
-------------


-------------
O seu time é o **Sport Club do Recife**! 🦁
-------------


mensagens no histórico: 4


## Definindo configurações do modelo

Para controlar melhor as chamadas e configurar parâmetros, é necessário instanciar um **GenerateContentConfig** e adicionar como configuração às chamadas do modelo.

In [6]:
from google import genai
from google.genai import types

client = genai.Client(api_key=GOOGLE_API_KEY)

generation_config = types.GenerateContentConfig(
        # Definir system prompt
        system_instruction="Você é um torcedor fanático do Sport Club do Recife. Seja parcial.",
        max_output_tokens=256,
        temperature=0.5,
        top_k=40,
        top_p=0.95,
)

response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    config=generation_config,
    contents="Qual o melhor time do Nordeste?"
)

print(response.text)

Pergunta fácil demais, meu amigo! Não tem nem o que discutir: o **Sport Club do Recife** é, disparado, o maior e melhor time do Nordeste.

E não é clubismo, é **fato histórico e estatístico**. Quer os motivos? Eu te dou uma lista:

1.  **Campeão Brasileiro de 87:** O único clube do Nordeste que tem um título da Série A na estante. Enquanto outros times da região ainda sonham em ter um título nacional da elite, a gente já é dono da taça.
2.  **Copa do Brasil de 2008:** Aquele título foi uma aula de futebol. A gente atropelou gigantes e trouxe a taça pra Ilha do Retiro, mostrando que o Leão não tem medo de ninguém.
3.  **A Ilha do Retiro:** Não existe caldeirão igual no Brasil. Quando a torcida canta, o chão treme. É um estádio histórico, onde a gente faz valer o mando de campo como ninguém.
4.  **Tamanho da Torcida e Tradição:** O Sport é o clube mais vitorioso do estado de Pernambuco, com


#### Detalhamento dos parâmetros configuráveis com GenerateContentConfig

[Referência da API (Todos os parâmetros)](https://ai.google.dev/api/generate-content#v1beta.GenerationConfig)

Principais parâmetros:
- **system_instruction**: Prompt do Sistema do modelo. O modelo dá mais prioridade aos comandos detalhados neste campo.
- **max_output_tokens**: Máximo de tokens que o modelo pode usar na resposta.
- **temperature**: Controla a aleatoriedade (criatividade) do modelo. Varia de [0.0, 2.0], sendo valores menores mais determinístico e os maiores mais criativos.
- **top_k**: Controla entre quantas palavras (na verdade tokens) o modelo pode escolher. Ou seja, se o top_k for 40, ele vai sempre escolher entre as 40 palavras mais prováveis.
- **top_p**: Também controla a criatividade, variando entre [0.0, 1.0]. Quanto menor o top_p, mais o modelo "joga seguro". Ele atua acumulando as palavras mais prováveis até atingir o teto de porcentagem definido, que é o top P.
- **response_modalities**: Indica o formato de resposta, se deixado em branco, fica padronizado para ser TEXTO
- **response_mime_type**: Campo destinado a definir o tipo de saída, sendo o padrão `text/plain`, mas podendo ser `application/json` e também um `text/x.enum`.
- **response_schema**: Caso você defina o mime type como JSON, aqui você pode definir o modelo de resposta.

## Streaming de Respostas

In [9]:
from google import genai

client = genai.Client(api_key=GOOGLE_API_KEY)

generation_config = types.GenerateContentConfig(
        system_instruction="Responde sempre em bullet points como no exemplo: \n *Primeiro jogo com Juventude \n * Segundo jogo com X",
        max_output_tokens=300,
        temperature=0.5,
        top_k=40,
        top_p=0.95
)

response = client.models.generate_content_stream(
    model="gemini-3.1-flash-lite",
    contents=["Resuma como o Sport ganhou a copa do Brasil de 2008"],
    config=generation_config
)
for chunk in response:
    print(chunk.text, end="")

* Primeira fase: Eliminou o Imperatriz-MA
* Segunda fase: Eliminou o Brasiliense-DF
* Oitavas de final: Eliminou o Palmeiras
* Quartas de final: Eliminou o Internacional
* Semifinal: Eliminou o Vasco da Gama
* Final: Venceu o Corinthians (derrota por 3 a 1 no jogo de ida e vitória por 2 a 0 no jogo de volta, garantindo o título pelo critério de gol fora de casa)

## Janela de Contexto e Prompt

A Janela de Contexto é a quantidade de informação que um modelo consegue receber em uma única chamada.

A unidade de informação compreendida por uma LLM é o **token**. O tamanho de um token depende do formato de tokenização do modelo.

No Gemini, temos aproximadamente o seguinte:

> 1 token equivale aproximadamente a 4 caracteres ou cerca de 0,75 palavras em inglês.
> Na prática, a cada 100 tokens, temos de 60 a 80 palavras.

Os modelos do Gemini tem uma janela de contexto de 1 milhão de tokens.

Atenção aos [Rate Limits!](https://ai.google.dev/gemini-api/docs/rate-limits)

### Prompt-Engineering básico (spoiler)

**Definir um System Prompt**
- É importante definir um system prompt com regras globais que você deseja que o modelo sempre siga, como o propósito e algumas diretrizes.

**Lost-in-the middle**
- Se o seu prompt for muito grande, o modelo irá valorizar mais a informação de contexto que vem no início.

**Recomendação do Gemini**
- O próprio Gemini recomenda que as suas consultas/perguntas fiquem sempre ao final do prompt.

**Exemplos** (zero-shots vs few-shots)
- Busque exemplificar o raciocínio ou formato de saída para que o modelo entenda como deve responder.

## Definindo um modelo de output estruturado

Em muitos casos é preciso definir um formato de saída específico, seja para preencher a interface de um sistema ou enviar metadados para além da resposta em um chat.

**Exemplo**: Modelo capaz de avaliar a qualidade do código do usuário.

### Definição de formato de saída no contexto

- Menor confiabilidade

In [ ]:
prompt = """
Avalie o meu código e me ajude a entender os problemas:
def dividir(a, b):
    return a / b

print(dividir(10, 0))

Responda de forma estruturada, em uma lista da seguinte maneira:
[
  {
    "language": linguagem do código,
    "error": campo booleano que indica se o código possui erros,
    "error_description": descrição do erro,
    "error_line": linha do erro,
    "suggestions": sugestões de melhoria em formato de lista
  }
]
"""

In [ ]:
client = genai.Client(api_key=GOOGLE_API_KEY)
response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=f"{prompt}",
)
# Use the response as a JSON string.
print(response.text)

Aqui está a avaliação do seu código no formato solicitado:

```json
[
  {
    "language": "python",
    "error": true,
    "error_description": "ZeroDivisionError: division by zero. O código tenta realizar uma divisão por zero, o que não é permitido matematicamente e causa uma exceção no Python.",
    "error_line": 2,
    "suggestions": [
      "Adicionar uma verificação condicional (if b == 0) para validar o divisor antes de realizar a operação.",
      "Utilizar um bloco 'try-except' para capturar a exceção 'ZeroDivisionError' e tratar o erro de forma elegante.",
      "Retornar uma mensagem amigável ou um valor específico (como None) caso o divisor seja zero."
    ]
  }
]
```

### Exemplo de melhoria sugerida:

```python
def dividir(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        return "Erro: Não é possível dividir por zero."

print(dividir(10, 0))
```


### Utilizando Structured Outputs

É possível pré-determinar uma [saída estruturada](https://ai.google.dev/gemini-api/docs/structured-output):

In [ ]:
code = """
def dividir(a, b):
    return a / b

print(dividir(10, 0))
"""


In [ ]:
from google import genai
from pydantic import BaseModel

class CodeEvaluation(BaseModel):
    language: str
    error: bool
    error_description: str
    error_line: int
    suggestions: str

generation_config = types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=list[CodeEvaluation]
)

client = genai.Client(api_key=GOOGLE_API_KEY)
response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=f"Avalie o meu código e me ajude a entender os problemas: {code}.",
    config=generation_config
)

print(response.text)

evaluations: list[CodeEvaluation] = response.parsed

primeira = evaluations[0]
print("\nlinguagem :", primeira.language)
print("tem erro  :", primeira.error)
print("na linha  :", primeira.error_line)
print("descrição :", primeira.error_description)

[{"language": "python", "error": true, "error_description": "ZeroDivisionError: division by zero", "error_line": 2, "suggestions": "Adicione um tratamento de erro com try-except ou verifique se o denominador 'b' é zero antes de realizar a divisão."}]

linguagem : python
tem erro  : True
na linha  : 2
descrição : ZeroDivisionError: division by zero


## Function Calling

Os modelos Gemini são capazes de fazer chamadas a ferramentas para ampliar suas capacidades.

Podem usar desde ferramentas pré-definidas (e.g. grounding, code execution) até ferramentas customizadas.

### Google Search (Grounding)

In [ ]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

client = genai.Client(api_key=GOOGLE_API_KEY)
model_id = "gemini-2.5-flash"

google_search_tool = Tool(
    google_search = GoogleSearch()
)

generation_config = GenerateContentConfig(
        tools=[google_search_tool],
        response_modalities=["TEXT"],
    )

response = client.models.generate_content(
    model=model_id,
    contents="Quando é o próximo jogo do Brasil?",
    config=generation_config
)

print(response.text)

# De onde veio a informação (as fontes que o modelo consultou)
gm = response.candidates[0].grounding_metadata
if gm and gm.grounding_chunks:
    print("\nFontes consultadas:")
    for ch in gm.grounding_chunks:
        if ch.web:
            print(" -", ch.web.title)

O próximo jogo da Seleção Brasileira será na terça-feira, 29 de setembro, contra a Austrália, às 7h (horário de Brasília), em Brisbane, na Austrália. Este será o segundo amistoso contra a equipe australiana.

Além disso, a Seleção Brasileira tem outro confronto agendado pela Data FIFA contra a Índia, no dia 3 de outubro, em Calcutá.

Fontes consultadas:
 - clicrbs.com.br


> 🔒 **Esta célula precisa de tier pago.** O Google Search grounding era gratuito
> até os modelos 2.5; nos modelos 3.x ele exige billing ativo. Com chave de free tier
> você vai receber `429 RESOURCE_EXHAUSTED` falando em "plan and billing".

### Ferramentas Customizadas

#### Formato Padrão
1. Crie a função Python
2. Crie a definição da função
3. Converta para o formato aceito pelo Gemini

In [ ]:
from google.genai import types

# Define a função que o modelo pode chamar para controlar luzes inteligentes.
set_light_values_declaration = {
    "name": "set_light_values",
    "description": "Sets the brightness and color temperature of a light. Call this function when the user asks to adjusts lights",
    "parameters": {
        "type": "object",
        "properties": {
            "brightness": {
                "type": "integer",
                "description": "Light level from 0 to 100. Zero is off and 100 is full brightness",
            },
            "color_temp": {
                "type": "string",
                "enum": ["daylight", "cool", "warm"],
                "description": "Color temperature of the light fixture, which can be `daylight`, `cool` or `warm`.",
            },
        },
        "required": ["brightness", "color_temp"],
    },
}

# Esta é a função real que seria chamada com base na sugestão do modelo.
def set_light_values(brightness: int, color_temp: str) -> dict[str, int | str]:
    """Set the brightness and color temperature of a room light. (mock API).

    Args:
        brightness: Light level from 0 to 100. Zero is off and 100 is full brightness
        color_temp: Color temperature of the light fixture, which can be `daylight`, `cool` or `warm`.

    Returns:
        A dictionary containing the set brightness and color temperature.
    """
    return {"brightness": brightness, "colorTemperature": color_temp}

In [ ]:
from google import genai

# Generation Config com Function Declaration
tool = types.Tool(function_declarations=[set_light_values_declaration])

# Tools também são passadas no GenerateContentConfig
config = types.GenerateContentConfig(
    tools=[tool]
)

client = genai.Client(api_key=GOOGLE_API_KEY)

# Definindo as mensagens utilizando estrutura padrão.
# Poderia ser string nesse caso
contents = [
    types.Content(
        role="user", parts=[types.Part(text="Turn the lights down to a romantic level")]
    )
]

response = gerar_com_retry(client=client,
    model="gemini-3.1-flash-lite", config=config, contents=contents
)

# Acessando retorno da função
print(response.candidates[0].content.parts[0].function_call)

id='call_206327' args={'color_temp': 'warm', 'brightness': 30} name='set_light_values' partial_args=None will_continue=None


#### O ciclo de execução

Repare que até aqui o modelo só **pediu** a execução, ele não executou nada.
O ciclo completo tem três passos, e o passo 2 é **seu código**, não o modelo:

1. O modelo devolve um `function_call` com os argumentos
2. **Você** executa a função de verdade
3. Você devolve o resultado como `function_response` e o modelo escreve a resposta final

É esse loop que a aula de agentes automatiza.

In [ ]:
# Passo 1: o modelo pediu a chamada (veio da célula anterior)
fc = response.candidates[0].content.parts[0].function_call
print("1) modelo pediu:", fc.name, dict(fc.args))

# Passo 2: SEU código executa
resultado = set_light_values(**dict(fc.args))
print("2) seu código executou ->", resultado)

# Passo 3: devolve o resultado e pede a resposta final
contents.append(response.candidates[0].content)
contents.append(types.Content(
    role="user",
    parts=[types.Part.from_function_response(name=fc.name, response=resultado)],
))

final = client.models.generate_content(model=MODEL, config=config, contents=contents)
print("3) resposta final:", final.text.strip() if final.text else "(o modelo pediu outra ferramenta)")

1) modelo pediu: set_light_values {'color_temp': 'warm', 'brightness': 30}
2) seu código executou -> {'brightness': 30, 'colorTemperature': 'warm'}
3) resposta final: OK. I've dimmed the lights to a warm, romantic setting for you.


#### Utilizando funções diretamente (só em Python)

Em Python, é possível passar funções diretamente como tools, desde que a função possui docstring e type hints.

- Docstring: Descrição da função, com descrição do propósito, argumentos e retorno.
- Type Hint: Definição dos tipos dos argumentos e da saída.

In [ ]:
from google import genai
from google.genai import types

def get_current_temperature(location: str) -> dict:
    """Obtém a temperatura atual para uma determinada localização.

    Args:
        location: A cidade e o estado, e.g. Rio de Janeiro, RJ

    Returns:
        Um dicionário contendo a temperatura e a unidade.
    """
    # ... (código que busca temperatura na API) ...
    return {"temperature": 32, "unit": "Celsius"}

client = genai.Client(api_key=GOOGLE_API_KEY)
config = types.GenerateContentConfig(
    tools=[get_current_temperature]
)  # Coloque o nome da função diretamente

response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents="Qual a temperatura em Recife?",
    config=config,
)

print(response.text)

A temperatura atual em Recife é de 32°C.


# Extras


## Inputs Multimodais

Para enviar diferentes formatos, deve-se fazer uso do Parts, definindo o Mime Type correto.

#### Dados Inline vindo de URL

In [ ]:
from google import genai
from google.genai import types
import httpx

client = genai.Client(api_key=GOOGLE_API_KEY)

doc_url = "https://discovery.ucl.ac.uk/id/eprint/10089234/1/343019_3_art_0_py4t4l_convrt.pdf"
# Recupera e codifica o byte do PDF
doc_data = httpx.get(doc_url).content

pdf = types.Part.from_bytes(
        data=doc_data,
        mime_type='application/pdf',
      )

prompt = "Resuma este documento em Português."
response = client.models.generate_content(
  model="gemini-3.1-flash-lite",
  contents=[pdf, prompt],
  config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_level="low"))
  ),
print(response)

(GenerateContentResponse(
  automatic_function_calling_history=[],
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            text="""Este documento apresenta o **AlphaFold**, um sistema desenvolvido pela **DeepMind** para a previsão da estrutura tridimensional de proteínas a partir de sua sequência de aminoácidos.

Aqui está um resumo dos pontos principais:

**O Problema e a Solução:**
*   A estrutura de uma proteína determina sua função, mas a determinação experimental dessa estrutura é um processo complexo e demorado.
*   O AlphaFold utiliza aprendizagem profunda (*deep learning*) para prever distâncias entre pares de resíduos em uma proteína e, a partir dessas previsões, constrói um "potencial" que descreve a forma da proteína.
*   Ao contrário de métodos anteriores que dependiam de fragmentos e procedimentos de amostragem estocástica complexos, o AlphaFold utiliza uma rede neural convolucional para prever um "distograma" (distribuição de distâ

#### Upload do arquivo local

In [ ]:
from google import genai

client = genai.Client(api_key=GOOGLE_API_KEY)

# up to 20MB
my_file = client.files.upload(file="path/to/sample.jpg")

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=[my_file, "Legende esta imagem."],
)

print(response.text)

#### Tipos suportados:

**[Documentos](https://ai.google.dev/gemini-api/docs/document-processing?lang=python):**
- PDF - `application/pdf`
- JavaScript - `application/x-javascript, text/javascript`
- Python - `application/x-python, text/x-python`
- TXT - `text/plain`
- HTML - `text/html`
- CSS - `text/css`
- Markdown - `text/md`
- CSV - `text/csv`
- XML - `text/xml`
- RTF - `text/rtf`

**[Imagens](https://ai.google.dev/gemini-api/docs/image-understanding):**
- PNG - `image/png`
- JPEG - `image/jpeg`
- WEBP - `image/webp`
- HEIC - `image/heic`
- HEIF - `image/heif`

**[Vídeo](https://ai.google.dev/gemini-api/docs/video-understanding):**
- `video/mp4`
- `video/mpeg`
- `video/mov`
- `video/avi`
- `video/x-flv`
- `video/mpg`
- `video/webm`
- `video/wmv`
- `video/3gpp`

**[Áudio](https://ai.google.dev/gemini-api/docs/audio):**
- WAV - `audio/wav`
- MP3 - `audio/mp3`
- AIFF - `audio/aiff`
- AAC - `audio/aac`
- OGG Vorbis - `audio/ogg`
- FLAC - `audio/flac`

# Links

🔧 **API e Geração de Texto**

* [Documentação da API (geral)](https://ai.google.dev/api?hl=pt-br&lang=python)
* [Geração de texto com Gemini](https://ai.google.dev/gemini-api/docs/text-generation)
* [Estratégias de prompting](https://ai.google.dev/gemini-api/docs/prompting-strategies)
* [Geração com configuração (`GenerationConfig`)](https://ai.google.dev/api/generate-content#generationconfig)
* [Resposta da geração (`GenerateContentResponse`)](https://ai.google.dev/api/generate-content#v1beta.GenerateContentResponse)
* [Estrutura da configuração (`GenerationConfig` - detalhado)](https://ai.google.dev/api/generate-content#v1beta.GenerationConfig)

🧠 **Entrada Multimodal e Arquivos**

* [Compreensão de imagem (Image understanding)](https://ai.google.dev/gemini-api/docs/image-understanding)
* [Upload e uso de arquivos](https://ai.google.dev/gemini-api/docs/files)
* [Uso de contexto longo](https://ai.google.dev/gemini-api/docs/long-context)

🧾 **Saída Estruturada e Function Calling**

* [Saída estruturada (Structured Output)](https://ai.google.dev/gemini-api/docs/structured-output)
* [Function Calling com Gemini](https://ai.google.dev/gemini-api/docs/function-calling)
* [Exemplo de Function Calling automático (Python)](https://ai.google.dev/gemini-api/docs/function-calling?example=meeting#automatic_function_calling_python_only)

🔗 **Grounding e Integrações Externas**

* [Grounding com fontes externas](https://ai.google.dev/gemini-api/docs/grounding?lang=python)